In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from collections import Counter

In [2]:
main_table_path = "../data/raw/application_train.csv"
df = pd.read_csv(main_table_path, header=0)

In [3]:
df_id = df['SK_ID_CURR']
df = df.drop(columns=['SK_ID_CURR'], axis=1)

In [4]:
def transforming_data(df):
    """
    Function only designed for lightgbm algorithm that accepts null values
    """
    print(f"Cols Count before: {len(df.columns)}")
    
    # Missing goods price flag
    df['GOODS_PRICE_MISSING'] = df['AMT_GOODS_PRICE'].isna().astype(int)
    
    # Adding the missing flag for EXT_SOOURCES
    df['EXT_SOURCE_1_MISSING_FLAG'] =  df['EXT_SOURCE_1'].isna().astype(int)
    df['EXT_SOURCE_2_MISSING_FLAG'] =  df['EXT_SOURCE_2'].isna().astype(int)
    df['EXT_SOURCE_3_MISSING_FLAG'] =  df['EXT_SOURCE_3'].isna().astype(int)
    
    # Imputing with median
    df['EXT_SOURCE_1'] = df['EXT_SOURCE_1'].fillna(df['EXT_SOURCE_1'].median(skipna=True))
    df['EXT_SOURCE_2'] = df['EXT_SOURCE_2'].fillna(df['EXT_SOURCE_2'].median(skipna=True))
    df['EXT_SOURCE_3'] = df['EXT_SOURCE_3'].fillna(df['EXT_SOURCE_3'].median(skipna=True))
    
    # Stat methods per record
    df['EXT_SOURCE_MEAN'] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].mean(axis=1)
    df['EXT_SOURCE_MEDI'] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].median(axis=1)
    df['EXT_SOURCE_MIN'] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].min(axis=1)
    df['EXT_SOURCE_MAX'] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].max(axis=1)
    
    
    ### Housing related cols
    housing_related_cols = ['APARTMENTS_AVG','BASEMENTAREA_AVG','YEARS_BEGINEXPLUATATION_AVG','YEARS_BUILD_AVG','COMMONAREA_AVG','ELEVATORS_AVG','ENTRANCES_AVG','FLOORSMAX_AVG','FLOORSMIN_AVG','LANDAREA_AVG','LIVINGAPARTMENTS_AVG','LIVINGAREA_AVG','NONLIVINGAPARTMENTS_AVG','NONLIVINGAREA_AVG','APARTMENTS_MODE','BASEMENTAREA_MODE','YEARS_BEGINEXPLUATATION_MODE','YEARS_BUILD_MODE','COMMONAREA_MODE','ELEVATORS_MODE','ENTRANCES_MODE','FLOORSMAX_MODE','FLOORSMIN_MODE','LANDAREA_MODE','LIVINGAPARTMENTS_MODE','LIVINGAREA_MODE','NONLIVINGAPARTMENTS_MODE','NONLIVINGAREA_MODE','APARTMENTS_MEDI','BASEMENTAREA_MEDI','YEARS_BEGINEXPLUATATION_MEDI','YEARS_BUILD_MEDI','COMMONAREA_MEDI','ELEVATORS_MEDI','ENTRANCES_MEDI','FLOORSMAX_MEDI','FLOORSMIN_MEDI','LANDAREA_MEDI','LIVINGAPARTMENTS_MEDI','LIVINGAREA_MEDI','NONLIVINGAPARTMENTS_MEDI','NONLIVINGAREA_MEDI','FONDKAPREMONT_MODE','HOUSETYPE_MODE','TOTALAREA_MODE','WALLSMATERIAL_MODE','EMERGENCYSTATE_MODE']
    for col in housing_related_cols:
        df[f'{col}_MISSING_FLAG'] = df[col].isna().astype(int)
    
    ### Social Circle cols
    # Separating the cat (product) 
    df['REVOLVING_LOAN'] = (df['NAME_CONTRACT_TYPE'] == 'Revolving loans').astype(int)
    
    
    ### Bureau related cols
    cols_bureau = ['AMT_REQ_CREDIT_BUREAU_HOUR','AMT_REQ_CREDIT_BUREAU_DAY','AMT_REQ_CREDIT_BUREAU_WEEK','AMT_REQ_CREDIT_BUREAU_MON','AMT_REQ_CREDIT_BUREAU_QRT','AMT_REQ_CREDIT_BUREAU_YEAR']
    for col in cols_bureau:
        df[f'{col}_MISSING_FLAG'] = df[col].isna().astype(int)

    ## Transforming dtypes en cats pour les objs
    categorical_cols = df.select_dtypes(include="object").columns

    for col in categorical_cols:
        df[col] = df[col].astype("category")
    
    
    print(f"Cols Count before: {len(df.columns)}")
    
    return df

In [5]:
n_df = transforming_data(df)

Cols Count before: 121
Cols Count before: 183


In [62]:
# Variables explicatives et cible
X = df.drop(columns=["TARGET"])
y = df["TARGET"]

# Découpage train / test
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

counter = Counter(y_train)
scale_pos_weight = counter[0] / counter[1]

# Modèle de classification
model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight
)

model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(200, verbose=True)]
)

[LightGBM] [Info] Number of positive: 19876, number of negative: 226132
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.023028 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 12515
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 177
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080794 -> initscore=-2.431606
[LightGBM] [Info] Start training from score -2.431606
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[1]	valid_0's auc: 0.726236	valid_0's binary_logloss: 0.275137


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,1000
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [63]:
for i in [0.05,0.1,0.2, 0.3, 0.5, 0.6, 0.8]:
    print(f"------> {i}")
    valid_pred = model.predict_proba(X_valid)[:, 1]
    y_pred = (valid_pred >= i).astype(int)

    print("AUC:", roc_auc_score(y_valid, valid_pred))
    print("Precision:", precision_score(y_valid, y_pred))
    print("Recall:", recall_score(y_valid, y_pred))
    print("F1:", f1_score(y_valid, y_pred))


------> 0.05
AUC: 0.7262358065923085
Precision: 0.08046761946571712
Recall: 1.0
F1: 0.1489496177692169
------> 0.1
AUC: 0.7262358065923085
Precision: 0.13083756783850115
Recall: 0.7648009698929077
F1: 0.22344884585866934
------> 0.2
AUC: 0.7262358065923085
Precision: 0.0
Recall: 0.0
F1: 0.0
------> 0.3


/home/adnane/miniconda3/envs/torch/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/adnane/miniconda3/envs/torch/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/adnane/miniconda3/envs/torch/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result

AUC: 0.7262358065923085
Precision: 0.0
Recall: 0.0
F1: 0.0
------> 0.5
AUC: 0.7262358065923085
Precision: 0.0
Recall: 0.0
F1: 0.0
------> 0.6
AUC: 0.7262358065923085
Precision: 0.0
Recall: 0.0
F1: 0.0
------> 0.8
AUC: 0.7262358065923085
Precision: 0.0
Recall: 0.0
F1: 0.0


/home/adnane/miniconda3/envs/torch/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [64]:
y_pred_train = model.predict(X_train)
acc_train = (y_train == y_pred_train).sum()/len(y_train)
acc_validation = (y_valid == y_pred).sum()/len(y_valid)

print(acc_train)
print(acc_validation)

0.919205879483594
0.9195323805342829


# TEST

In [65]:
main_table_path = "../data/raw/application_test.csv"
t_df = pd.read_csv(main_table_path, header=0)

In [66]:
t_df = transforming_data(t_df)
t_df_id = t_df['SK_ID_CURR']
t_df = t_df.drop(columns=['SK_ID_CURR'], axis=1)
t_pred = model.predict(t_df)

Cols Count before: 121
Cols Count before: 183


In [67]:

submission = pd.DataFrame(
    {
        'SK_ID_CURR': t_df_id.values,
        'TARGET': t_pred
    }
)

In [68]:
submission.to_csv('../data/submission.csv', index=False)

In [69]:
submission

,SK_ID_CURR,TARGET
0,100001,0
1,100005,0
2,100013,0
3,100028,0
4,100038,0
...,...,...
48739,456221,0
48740,456222,0
48741,456223,0
48742,456224,0


In [70]:
submission['TARGET'].value_counts()

TARGET
0    48744
Name: count, dtype: int64

In [71]:
len(t_df)

48744

for i, col in enumerate(df.columns):
    missing = df[col].isna().sum()
    pct = missing / len(df)
    dt = df[col].dtype
    print(f"{i:<5} {col:<35} {missing:<15} {pct:<15.2%} {str(dt):<15} {'-':<2}")

In [72]:
for i, col in enumerate(df.columns):
    missing = df[col].isna().sum()
    pct = missing / len(df)
    dt = df[col].dtype
    print(f"{i:<5} {col:<35} {missing:<15} {pct:<15.2%} {str(dt):<15} {'-':<2}")

0     TARGET                              0               0.00%           int64           - 
1     NAME_CONTRACT_TYPE                  0               0.00%           category        - 
2     CODE_GENDER                         0               0.00%           category        - 
3     FLAG_OWN_CAR                        0               0.00%           category        - 
4     FLAG_OWN_REALTY                     0               0.00%           category        - 
5     CNT_CHILDREN                        0               0.00%           int64           - 
6     AMT_INCOME_TOTAL                    0               0.00%           float64         - 
7     AMT_CREDIT                          0               0.00%           float64         - 
8     AMT_ANNUITY                         12              0.00%           float64         - 
9     AMT_GOODS_PRICE                     278             0.09%           float64         - 
10    NAME_TYPE_SUITE                     1292            0.42%       